# Olist Data Quality — Notebook educativo

## Fases 1 a 5: Entorno, carga RAW y diagnóstico integral

**Objetivo:** verificar el entorno; importar CSV originales a RAW sin alterar su contenido; comprender claves, relaciones y granularidad; y generar un reporte **inicial** de calidad con SQL DuckDB.

**Aún no:** ejecutar las 15 reglas de transformación RAW → CLEAN, publicar Parquet o afirmar que los datos ya están limpios.

Guías: `docs/FASE_1_TUTORIAL.md`, `docs/FASE_2_TUTORIAL.md`, `docs/FASE_3_TUTORIAL.md`.

### Arquitectura fija

```text
CSV originales -> Extracción Python + DuckDB -> RAW
              -> 15 reglas SQL -> CLEAN
              -> pruebas críticas -> Parquet + reporte de calidad
```

RAW y CLEAN son las únicas capas persistentes de datos. `audit` será exclusivamente un esquema de control, no una capa STAGING.

### 1. Detectar la carpeta raíz y mostrar el entorno

En VS Code, selecciona **Python (olist-quality)** arriba a la derecha. El bloque siguiente localiza la raíz incluso si el directorio de trabajo es `notebooks/`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((candidate for candidate in (cwd, *cwd.parents)
                     if (candidate / "environment.yml").exists()), None)
if project_root is None:
    raise RuntimeError("Abre el notebook desde la carpeta olist-data-quality")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Raíz:", project_root)
print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)

### 2. Comprobar bibliotecas

Verificamos las versiones realmente instaladas. Si una importación falla, vuelve al Paso 4 del tutorial.

In [ ]:
import duckdb
import pandas as pd
import pyarrow as pa
import matplotlib
print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Matplotlib:", matplotlib.__version__)

### 3. Crear la base persistente

El módulo `src.bootstrap` es **la única implementación** de esta operación; el notebook sólo lo invoca. Repetir la celda no borra datos.

In [ ]:
from src.bootstrap import bootstrap
from src.config import DB_PATH

print("Base de datos:", bootstrap())
connection = duckdb.connect(str(DB_PATH))
try:
    schema_rows = connection.execute("""
        SELECT schema_name FROM information_schema.schemata
        WHERE schema_name IN ('raw', 'clean', 'audit') ORDER BY schema_name
    """).fetchall()
    print("Esquemas:", schema_rows)
    assert {'raw', 'clean', 'audit'}.issubset({row[0] for row in schema_rows})
finally:
    connection.close()

### 4. Verificar archivos originales

Los nueve archivos oficiales deben copiarse a `data/raw/` sin cambios. Esta celda funciona antes de descargarlos: reporta los faltantes y no inventa cantidades.

In [ ]:
from src.config import DATA_RAW_DIR, EXPECTED_CSV_FILES

present = sorted(path.name for path in DATA_RAW_DIR.glob('*.csv'))
missing = sorted(set(EXPECTED_CSV_FILES) - set(present))
print("CSV encontrados:", len(present))
print("CSV faltantes:", missing)
if missing:
    print("Descarga el dataset Olist siguiendo el Paso 7 del tutorial.")

### 5. Inspeccionar el origen y generar reportes

**Sólo cuando estén los nueve archivos** se ejecuta el inventario completo. Se cuentan registros CSV (no líneas físicas) y se registra una huella SHA-256 que ayuda a verificar inmutabilidad. Los archivos originales no se modifican.

In [ ]:
from src.source_inventory import build_inventory

if missing:
    print("Inventario pendiente: primero descargar los 9 CSV originales.")
else:
    inventory = build_inventory()
    inventory_df = pd.DataFrame(inventory)
    display(inventory_df[["file", "records_count", "columns_count", "row_width_mismatches"]])
    print("Inventario JSON y CSV creado en reports/quality_reports/")

### 6. Comprobar integridad de estructura sin convertir tipos

Los errores de ancho en un CSV se reportan para su investigación. Una fila irregular **no implica automáticamente** que deba eliminarse.

In [ ]:
if missing:
    print("Validación del ancho pendiente: faltan archivos originales.")
else:
    width_issues = inventory_df.loc[inventory_df["row_width_mismatches"] > 0,
                                     ["file", "row_width_mismatches"]]
    if width_issues.empty:
        print("No se observaron filas con cantidad de campos distinta al encabezado.")
    else:
        print("Archivos con anomalías estructurales:")
        display(width_issues)

### 7. Final de Fase 1

Una Fase 1 verificada tendrá: entorno Conda, kernel correcto, `database/ecommerce.duckdb`, nueve archivos Olist en `data/raw/` y reportes de inventario. **Aún no se han creado tablas RAW, CLEAN ni Parquet.**

La Fase 2 usará las columnas reales para determinar PK, FK, claves de negocio, granularidad y relaciones sin suponer que todos los duplicados o nulos son errores.

## Fase 2 — Comprensión de Olist y carga de CSV a RAW

Los CSV originales se preservan. En este capítulo se crean las tablas RAW con todas sus columnas como `VARCHAR`, sin limpieza. El perfilado posterior revela datos faltantes, claves candidatas y relaciones reales; **no se deduplican pedidos, artículos, pagos ni registros geográficos**.

### 8. Claves y granularidad: hipótesis que verificaremos

Una fila en `orders` representa un pedido, en `order_items` representa un artículo de pedido y en `order_payments` un pago. `customer_id` y `customer_unique_id` son diferentes; `review_id` y prefijos postales **no se dan por únicos**. Estas hipótesis del catálogo se contrastan con los datos cargados.

In [ ]:
from src.model_metadata import TABLE_SPECS, RELATIONSHIPS
for spec in TABLE_SPECS:
    print(f"{spec.name:20} | {spec.grain} | clave candidata: {spec.candidate_key or 'ninguna'}")

### 9. Carga RAW transaccional

Python coordina la lectura, DuckDB ejecuta la importación SQL. Se comprueban SHA-256, encabezados, tipos `VARCHAR` y conteos contra el inventario. No se omiten errores de lectura ni se alteran los CSV.

In [ ]:
from src.extract import load_raw
if missing:
    raise FileNotFoundError("No se puede cargar RAW: faltan los CSV indicados en la Fase 1")
raw_load = load_raw()
display(pd.DataFrame(raw_load)[["table", "rows", "columns", "action"]])

### 10. Perfilado inicial, sin modificar RAW

Se generan reportes de columnas, nulos/vacíos, unicidad de claves candidatas, referencias entre tablas, rangos de fechas, valores monetarios y categorías. Los indicadores **son observaciones reales**, no se supone que deban ser cero.

In [ ]:
from src.understand import profile_raw
profile = profile_raw()
for name in ("raw_tables", "raw_keys", "raw_references", "raw_dates", "raw_monetary"):
    print("\n", name)
    display(pd.DataFrame(profile[name]).head(30))

### 11. Porcentaje de datos faltantes por tabla

La figura resume el porcentaje de campos vacíos o nulos detectados en RAW. Un valor ausente no siempre constituye un error de negocio; requiere interpretación contextual.

In [ ]:
import matplotlib.pyplot as plt
missing_df = pd.DataFrame(profile["raw_missingness"])
if not missing_df.empty:
    summary = missing_df.groupby("table", as_index=False)["missing_or_empty_pct"].mean().sort_values("missing_or_empty_pct")
    ax = summary.plot.barh(x="table", y="missing_or_empty_pct", legend=False, figsize=(10, 5))
    ax.set_title("Promedio de porcentaje de valores ausentes por columna en RAW")
    ax.set_xlabel("Porcentaje medio de celdas ausentes")
    ax.set_ylabel("Tabla RAW")
    plt.tight_layout()
    plt.show()

### 12. Verificar conteos CSV vs RAW y revisar la propuesta ER

La cantidad de filas debe coincidir; una diferencia es una falla de la extracción. En el diagrama ER, las claves se señalan como PK **sólo si la prueba de unicidad y completitud pasó**. Las líneas representan relaciones conceptuales, no constraints físicos en RAW.

In [ ]:
source_counts = {row["file"]: row["records_count"] for row in inventory}
comparison = pd.DataFrame(profile["raw_tables"]).copy()
comparison["csv_rows"] = comparison["source"].map(source_counts)
comparison["same_rows"] = comparison["records_count"] == comparison["csv_rows"]
display(comparison[["table", "csv_rows", "records_count", "same_rows"]])
assert comparison["same_rows"].all(), "Los conteos de CSV y RAW difieren"
print("Diagrama ER:", project_root / "reports/data_dictionary/olist_er.mmd")

### 13. Resultado de Fase 2

Hemos inspeccionado los CSV y cargado sus valores en RAW sin convertir ni limpiar. Se crearon archivos de diagnóstico y un diccionario generado a partir de columnas reales. **La siguiente Fase 3** analiza los problemas y prepara el reporte de calidad inicial; **Fase 4** implementará las 15 reglas y construirá CLEAN. No afirmar que existen Parquet o una capa CLEAN procesada hasta ejecutarlo y validarlo.

---

# Fase 3 — Diagnóstico integral de calidad antes de limpiar

En esta fase cuantificaremos los problemas en las nueve tablas `raw`, sin crear una capa STAGING ni modificar RAW. Los diagnósticos son **evidencia para elegir una política**, no instrucciones de eliminación automática.

**Pasos:** 1) ejecutar perfilado, 2) inspeccionar tablas, 3) estudiar pagos y valores atípicos, 4) revisar informes y figuras, 5) interpretar las limitaciones. Todas las cifras saldrán de los CSV reales que cargaste en Fase 2.

### 14. Diagnóstico completo sobre RAW

**¿Dónde?** Esta celda Jupyter llama a la misma implementación que el comando `python -m src.diagnose`.

**¿Por qué?** Para crear una línea base reproducible de faltantes, duplicados, fechas, dinero, referencias y estados del pedido. La base DuckDB se abre en modo **solo lectura**.

In [ ]:
from src.diagnose import diagnose_raw
phase3 = diagnose_raw()
print("Tablas evaluadas:", len(phase3["overview"]))
print("Tipos de alerta cuantificados:", len(phase3["issues"]))
display(pd.DataFrame(phase3["overview"]))

### 15. Indicadores de faltantes y duplicados

`NULL`, cadenas vacías y textos compuestos únicamente por espacios se contabilizan **por separado**; una ausencia puede ser legítima para un pedido no entregado. La duplicación exacta de filas y la repetición de `order_id` no son sinónimos.

In [ ]:
missing3 = pd.DataFrame(phase3["missingness"])
missing3 = missing3.sort_values("missing_or_empty_pct", ascending=False)
display(missing3.head(20))
display(pd.DataFrame(phase3["duplicates"]))
display(pd.DataFrame(phase3["keys"]))

### 16. Referencias y conversiones de tipos

Las claves son *candidatas*, no restricciones físicas del CSV. Una relación con catálogo de traducción puede ser opcional. Un intento de conversión fallido exige revisión antes de sustituir o descartar el dato.

In [ ]:
display(pd.DataFrame(phase3["references"]))
display(pd.DataFrame(phase3["conversions"]))
display(pd.DataFrame(phase3["postals"]))

### 17. Análisis temporal y de precios

El método IQR usa Q1, Q3 y 1.5×IQR para **detectar posibles valores extremos**, sin etiquetarlos automáticamente como errores. La entrega posterior a la fecha prevista puede representar una demora normal.

In [ ]:
display(pd.DataFrame(phase3["temporal"]))
display(pd.DataFrame(phase3["monetary_issues"]))
display(pd.DataFrame(phase3["outliers"]))

### 18. Conciliación de artículos, fletes y pagos

**Riesgo crítico de SQL:** si un pedido tiene dos artículos y dos pagos y hacemos un `JOIN` directo, se generan cuatro filas, lo que infla ambos importes. Nuestra consulta agrupa artículos y pagos por pedido por separado **antes** de unirlos.

Los vouchers, descuentos, estados pendientes y variaciones de redondeo requieren análisis antes de corregir importes.

In [ ]:
display(pd.DataFrame(phase3["payment_summary"]))
display(pd.DataFrame(phase3["payment_details"]).head(30))

### 19. Priorización y reporte inicial

Las incidencias de distintas reglas pueden solaparse: **no sumes sus cantidades para obtener el número total de filas inválidas**. Aquí tampoco podemos certificar exactitud sin comparar contra una fuente independiente.

In [ ]:
display(pd.DataFrame(phase3["issues"]).head(40))
display(pd.DataFrame(phase3["kpis"]))
from src.config import QUALITY_REPORTS_DIR, FIGURES_DIR
print("Reporte inicial:", QUALITY_REPORTS_DIR / "phase3_quality_report.md")
print("Resultados CSV:", QUALITY_REPORTS_DIR / "phase3_*.csv")
print("Gráficos:", FIGURES_DIR / "phase3_*.png")

### 20. Verificar los resultados y cerrar la Fase 3

**Comprobaciones:** existen CSV de diagnóstico, manifiesto y reporte Markdown; las nueve tablas RAW siguen presentes. Las pruebas del repositorio se ejecutan con `python -m pytest -q tests`.

**Conclusión:** el diagnóstico inicial permite discutir el tratamiento de cada una de las 15 reglas de limpieza. En la **Fase 4** las implementaremos mediante SQL DuckDB directamente de RAW a CLEAN y registraremos sus efectos sin destruir el origen.

In [ ]:
from src.config import DB_PATH
assert (QUALITY_REPORTS_DIR / "phase3_quality_report.md").is_file()
assert (QUALITY_REPORTS_DIR / "phase3_summary.json").is_file()
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    print("RAW:", con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='raw'").fetchone()[0], "tablas")
    print("CLEAN:", con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='clean'").fetchone()[0], "tablas (no modificadas en Fase 3)")
print("Fase 3: artefactos generados; revisar su contenido y resultados de tests.")

---
# Fase 4 — Transformación de RAW a CLEAN

En las Fases 1–3 preparamos el entorno, importamos los nueve CSV sin transformarlos y calculamos una línea base de problemas. **Ahora construiremos CLEAN sin modificar RAW.** El flujo será: SQL por tabla → clasificación por regla → cuarentena/auditoría → CLEAN → validaciones preliminares. No hay STAGING persistente y **no se exporta Parquet todavía**.

Antes de ejecutar revisa `docs/FASE_4_TUTORIAL.md`, especialmente el tratamiento de claves, campos faltantes y anomalías de negocio. La rutina es la misma que usa la terminal: no se duplica la lógica en celdas sueltas.

In [ ]:
from src.transform import transform_data
from src.config import DB_PATH, QUALITY_REPORTS_DIR

fase4 = transform_data(DB_PATH, QUALITY_REPORTS_DIR)
print("Run ID:", fase4['run_id'])
print("Reporte:", fase4['report'])
assert fase4['status'] == 'COMPLETED'


## Resultados reales calculados en tu base

Los conteos mostrados en estas tablas provienen de tu RAW y CLEAN. Verifica que por cada tabla `raw_rows == clean_rows + quarantined_rows`; que haya más de un artículo o pago por pedido es normal. Las detecciones por regla se pueden solapar; no se deben sumar como si fueran filas distintas.

In [ ]:
import pandas as pd
from IPython.display import display

df_filas = pd.DataFrame(fase4['tables'])
df_reglas = pd.DataFrame(fase4['rules'])
display(df_filas)
display(df_reglas[['rule_id','table_name','detected','corrected','quarantined','unresolved','status']])
assert (df_filas['raw_rows'] == df_filas['clean_rows'] + df_filas['quarantined_rows']).all()


## Visualización: cantidad de filas en RAW y CLEAN

Esta gráfica **no demuestra automáticamente mejor calidad**: una reducción podría representar la eliminación legítima de redundancias o la cuarentena de registros que requieren revisión. Examina `phase4_quarantine_summary.csv` y la política de cada regla antes de interpretarla.

In [ ]:
import matplotlib.pyplot as plt
ax = (df_filas.set_index('table')[['raw_rows', 'clean_rows']]
      .plot(kind='bar', figsize=(12, 4), title='Olist: registros RAW vs CLEAN'))
ax.set_ylabel('Número de filas')
ax.set_xlabel('Tabla')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Auditoría y cuarentena

La siguiente consulta consulta exclusivamente los resultados de **esta ejecución**, filtrando por run_id, por lo que no mezcla datos históricos. Se registran los datos originales en JSON para una revisión posterior. No edites directamente `audit.quarantine_records`.

In [ ]:
import duckdb
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    resumen_quarantine = con.execute('''SELECT source_table, reason, COUNT(*) AS registros
    FROM audit.quarantine_records WHERE run_id = ?
    GROUP BY source_table, reason ORDER BY 1, 2''', [fase4['run_id']]).df()
    resumen_reglas = con.execute('''SELECT rule_id, status, COUNT(*) AS comprobaciones,
           SUM(detected) AS detectadas, SUM(unresolved) AS pendientes
    FROM audit.rule_results WHERE run_id = ?
    GROUP BY 1, 2 ORDER BY 1, 2''', [fase4['run_id']]).df()
display(resumen_quarantine)
display(resumen_reglas)


## Comprobación de reproducibilidad

La reconstrucción de CLEAN puede repetirse sin duplicar registros. El historial de auditoría sí crecerá en una fila por ejecución y por regla evaluada, porque documenta las nuevas comprobaciones. Para verificarla desde una terminal: `python -m src.transform` y `python -m pytest -q tests`.

**Conclusión:** RAW permanece inmutable, CLEAN contiene los registros tipados que superaron las comprobaciones críticas preliminares y las excepciones quedan registradas. En la **Fase 5** construiremos las pruebas formales y el bloqueo definitivo de publicación antes de generar Parquet.

---
# Fase 5 — Validación formal de CLEAN

**Objetivo:** comprobar la calidad de los resultados de Fase 4 con consultas SQL independientes y registrar PASS / FAIL / WARNING. Esta fase **no modifica RAW/CLEAN ni exporta Parquet**.

La severidad se justifica por el impacto empresarial: problemas críticos de esquema, claves, integridad, fechas imposibles o importes no válidos bloquean la aprobación; casos sospechosos como outliers, diferencias monetarias y categorías sin traducción generan advertencias revisables. Consulta el tutorial `docs/FASE_5_TUTORIAL.md`.

### 21. Ejecutar la misma validación que en terminal

La función `validate_clean()` funciona independientemente de las variables de las celdas anteriores. Asegúrate de haber ejecutado la Fase 4. El código imprime el resultado y genera CSV, JSON y Markdown en `reports/quality_reports/`.

In [ ]:
from src.validate import validate_clean
from src.config import DB_PATH, QUALITY_REPORTS_DIR
from IPython.display import display
import pandas as pd

fase5 = validate_clean(db_path=DB_PATH, report_dir=QUALITY_REPORTS_DIR)
print("Estado de la validación:", fase5["status"])
print("Pruebas críticas fallidas:", fase5["critical_failures"])
print("Pruebas con advertencias:", fase5["warning_checks"])
print("Informe generado:", fase5["report"])

checks5 = pd.DataFrame(fase5["checks"])
display(checks5[["check_id", "rule_id", "table_name", "severity", "status",
                 "violations", "population", "explanation"]].head(35))

### 22. Interpretar los controles que requieren atención

`FAIL` es una condición crítica que bloquea la futura publicación. `WARNING` es una excepción documentada que **no demuestra necesariamente un error**. Las cantidades son métricas por control y pueden solaparse; no sumarlas para inferir filas únicas afectadas.

In [ ]:
pendientes5 = checks5[checks5["status"].isin(["FAIL", "WARNING"])]
display(pendientes5[["check_id", "table_name", "status", "violations", "explanation"]]
        .sort_values(["status", "violations"], ascending=[True, False]).head(50))

print("Registros en la muestra de conciliación:", len(fase5["payment_exceptions"]))
display(pd.DataFrame(fase5["payment_exceptions"]).head(20))

### 23. Visualizar la distribución de resultados

Esta visualización representa la cantidad de pruebas por resultado, **no** la cantidad de registros afectados. El gráfico utiliza exclusivamente las pruebas calculadas por SQL.

In [ ]:
import matplotlib.pyplot as plt
conteos5 = checks5["status"].value_counts().reindex(["PASS", "WARNING", "FAIL"], fill_value=0)
ax = conteos5.plot(kind="bar", figsize=(7,4), rot=0, title="Controles de calidad por resultado")
ax.set_ylabel("Número de controles")
ax.set_xlabel("Resultado")
plt.tight_layout()
plt.show()

### 24. Auditar la ejecución y concluir

El validador conserva históricos por `validation_run_id`. Si `critical_failures > 0`, la fase fue evaluada pero la publicación queda **BLOQUEADA**; investigar antes de continuar. Si no hay fallos críticos, se puede seguir hacia la Fase 6 para orquestación. Todavía no publicar Parquet.

In [ ]:
import duckdb
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    historial5 = con.execute("""SELECT validation_run_id, transform_run_id,
                                     started_at, status, critical_failures,
                                     warning_checks
                              FROM audit.validation_runs
                              ORDER BY started_at DESC LIMIT 10""").df()
display(historial5)
assert (QUALITY_REPORTS_DIR / "phase5_validation_report.md").is_file()
print("Decisión:", "APROBADO PARA CONTINUAR" if fase5["critical_failures"] == 0 else
      "BLOQUEADO: consultar pruebas críticas antes de publicar")

---
# Fase 6 — Orquestación y revisión temporal

Esta sección reutiliza los mismos módulos del proyecto. Como ya se ejecutaron las fases anteriores en este notebook, **la validación de esta sección no reconstruye RAW ni CLEAN**. Para hacer una ejecución integral desde cero, utiliza la terminal: `python -m src.pipeline --no-figures`. La Fase 6 no publica Parquet.


### Paso 25. Ejecutar validación y revisión sin reprocesar fuentes

Los 189 eventos del análisis anterior permanecerán `PENDIENTE` si aún no existe un responsable que apruebe formalmente su tratamiento. Es correcto que el dictamen sea `BLOCKED`.


In [ ]:
from src.pipeline import run_pipeline
from src.config import DB_PATH, QUALITY_REPORTS_DIR

fase6 = run_pipeline(
    db_path=DB_PATH, report_dir=QUALITY_REPORTS_DIR, validate_only=True
)
print("Dictamen:", fase6["status"])
print("Reporte:", fase6["report"])


### Paso 26. Interpretar el dictamen y distinguir fallos originales y aceptaciones documentales

Un `FAIL` original de Fase 5 se conserva como evidencia. El reporte de Fase 6 permite comprobar cuántas incidencias siguen sin resolución y cuántas tienen una excepción formal válida.


In [ ]:
import pandas as pd
from IPython.display import display

resumen6 = fase6["gate"]
display(pd.DataFrame([
    {"Indicador": "Estado original de Fase 5", "Valor": resumen6["phase5_original_status"]},
    {"Indicador": "FAIL críticos originales", "Valor": resumen6["phase5_raw_critical_failed_checks"]},
    {"Indicador": "Eventos aceptados documentalmente", "Valor": resumen6["formally_accepted_temporal_events"]},
    {"Indicador": "Eventos críticos sin resolver", "Valor": resumen6["unresolved_critical_events"]},
    {"Indicador": "Dictamen Fase 6", "Valor": resumen6["status"]},
]))


### Paso 27. Consultar el historial de auditoría

La consulta siguiente es de solo lectura y permite comprobar trazabilidad por etapa. El historial crece entre ejecuciones aunque los datos de RAW y CLEAN no se dupliquen.


In [ ]:
import duckdb
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    historia6 = con.execute("""
      SELECT run_id, status, mode, critical_failures,
             unresolved_critical_events, started_at, finished_at
      FROM audit.pipeline_runs
      ORDER BY started_at DESC LIMIT 10
    """).df()
display(historia6)


### Paso 28. Comprobación final de la Fase 6

Revisa el informe `reports/quality_reports/phase6_pipeline_report.md`, las decisiones por evento y el CSV de revisiones. **No cambies fechas ni apruebes automáticamente anomalías**. Si el dictamen es `BLOCKED`, la automatización funcionó correctamente al impedir la publicación. La Fase 7 quedará condicionada a revisar todas las decisiones pendientes y las pruebas críticas.


## Fase 7 — Publicación Parquet condicionada a la calidad

La Fase 7 **no elimina ni oculta** las 189 incidencias temporales detectadas. La publicación de las nueve tablas CLEAN sólo se autoriza si la validación y el registro de decisiones cumplen la política establecida, y si existe autorización independiente cuando se pretenda conservar excepciones críticas formalmente aceptadas.

La siguiente celda ejecuta únicamente el **preflight** (sin Parquet). El comando de publicación real debe ejecutarse expresamente en la terminal, después de comprobar la autorización; no se incluye como celda de ejecución automática.

Comandos de terminal (desde la raíz del proyecto):

```powershell
python -m src.release
python -m src.release --publish
```

En el estado actual, ambas instrucciones deben devolver `BLOCKED`; la segunda no publica los datos.

In [ ]:
# Fase 7: verificar de nuevo CLEAN y el control de excepciones, SIN exportar.
from src.release import run_release
from src.config import DB_PATH, QUALITY_REPORTS_DIR, DATA_CLEAN_DIR
preflight = run_release(db_path=DB_PATH, report_dir=QUALITY_REPORTS_DIR,
                        clean_dir=DATA_CLEAN_DIR, publish=False)
print('Estado de publicación:', preflight['status'])
print('Parquet publicado:', preflight['published'])
for reason in preflight.get('reasons', []):
    print('Motivo de bloqueo:', reason)

### Verificación y límites

- El preflight genera `reports/quality_reports/phase7_preflight_report.md` y `.json` con IDs y motivos calculados.
- Únicamente después de un release aprobado aparecerán `data/clean/*.parquet`, `data/clean/release_manifest.json` y `reports/quality_reports/phase7_quality_report.md`.
- No se declara publicada la Fase 7 mientras el gate real permanezca `BLOCKED`.
- Para el detalle paso a paso, consultar `docs/FASE_7_TUTORIAL.md`.

## Cierre Fase 7 — Política temporal R05 y publicación trazable

Esta actualización resuelve el bloqueo por fechas del transportista *no verificables* sin inventar fechas ni eliminar pedidos. La fuente `raw.orders` permanece intacta; CLEAN usa `NULL` para el evento de transportista contradictorio y AUDIT conserva el valor original y el motivo. La vista `clean.orders_temporal_quality` diferencia entre `SOURCE_MISSING`, `QUALITY_SUPPRESSED` y `USABLE`. Revisa el tutorial `docs/FASE_7_CIERRE_POLITICA_TEMPORAL.md`.

**Ejecuta primero en la terminal** (no en el notebook):

```powershell
python -m pytest -q tests
python -m src.transform
python -m src.validate
python -m src.release
```

La publicación requiere `python -m src.release --publish`, exclusivamente si el preflight devuelve `ELIGIBLE`.


In [ ]:
from pathlib import Path
import json
import duckdb
from src.config import DB_PATH, DATA_CLEAN_DIR, QUALITY_REPORTS_DIR

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    print("Estado temporal de los pedidos CLEAN:")
    display(con.execute("""
        SELECT carrier_timestamp_quality_status, COUNT(*) AS pedidos
        FROM clean.orders_temporal_quality
        GROUP BY 1 ORDER BY 1
    """).df())
    print("Incidencias originales de R05 conservadas en auditoría:")
    display(con.execute("""
        SELECT issue_code, COUNT(*) AS eventos
        FROM audit.temporal_policy_events
        WHERE transform_run_id=(
            SELECT run_id FROM audit.transform_runs
            WHERE status='COMPLETED'
            ORDER BY finished_at DESC NULLS LAST, run_id DESC LIMIT 1)
        GROUP BY 1 ORDER BY 1
    """).df())

preflight_path = QUALITY_REPORTS_DIR / 'phase7_preflight_summary.json'
if preflight_path.is_file():
    preflight = json.loads(preflight_path.read_text(encoding='utf-8'))
    print('Preflight:', preflight['status'])
    print('Razones:', preflight.get('reasons', []))

manifest_path = DATA_CLEAN_DIR / 'release_manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('Último release:', manifest['release_id'])
    print('Parquet:', len(manifest['tables']))
    print('Eventos R05 en archivo de trazabilidad:', manifest['temporal_policy']['events'])
else:
    print('No hay release publicado: ejecutar --publish sólo después de preflight ELIGIBLE')
